# Lakelink Fiber Network Scenario Twin: synthetic data generator
The single generator notebook for all P1-GEN-* tasks (D-006, data_contracts.md section 0). Target `cdm_tmforum.network_scenario_twin`; tag `telco_project: network_scenario_twin`; seed 4471; run `nst-sim-001`.

Sections (one per task): P1-GEN-REFERENCE (done) | P1-GEN-TOPOLOGY | P1-GEN-SERVICE | P1-GEN-TELEMETRY | P1-GEN-OPS | P1-GEN-COPPER.

Rules: REUSE FIRST. FK values come only from existing parent tables. Company name is always Lakelink Fiber. Customer names are fictional.

In [0]:
import random, csv
from pyspark.sql import functions as F
CAT, SCH = "cdm_tmforum", "network_scenario_twin"
FQ = lambda t: f"{CAT}.{SCH}.{t}"
SEED, RUN_ID = 4471, "nst-sim-001"
ROOT = "/Workspace/Users/stephen.hage@databricks.com/network-scenario-twin"

def publish(table, df, table_comment, col_comments):
    """Overwrite a twin table, then apply table/column comments and governed tags."""
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(FQ(table))
    esc = lambda s: s.replace("'", "''")
    spark.sql(f"COMMENT ON TABLE {FQ(table)} IS '{esc(table_comment)}'")
    for c, v in col_comments.items():
        spark.sql(f"ALTER TABLE {FQ(table)} ALTER COLUMN {c} COMMENT '{esc(v)}'")
    spark.sql(f"ALTER TABLE {FQ(table)} SET TAGS ('telco_project'='network_scenario_twin','domain'='operations')")

LINEAGE = {"source_table": "Lineage: fully qualified parent table (null for SYNTHETIC).",
           "source_id": "Lineage: parent key value as string (null for SYNTHETIC).",
           "origin": "Data origin: REAL, MIXED or SYNTHETIC.",
           "simulation_run_id": "Generator run id (nst-sim-001)."}

In [0]:
# EXTEND tmf_resource.resource_specification: 20 non-deprecated transport specs, lowest ids first
type_plan = ["fiber_span"]*4 + ["optical_node"]*4 + ["site"]*2 + ["wavelength"]*4 + ["port"]*4 + ["och"]*2
src = (spark.table(f"{CAT}.tmf_resource.resource_specification")
       .where("status <> 'deprecated' AND max_bandwidth_mbps IS NOT NULL AND technology_domain IN ('sdh','mpls','ftth','fixed','gpon')")
       .orderBy("resource_specification_id").limit(20)
       .select("resource_specification_id", "max_bandwidth_mbps").collect())
assert len(src) == 20
label = {"fiber_span": "Long-Haul Fiber Span", "optical_node": "ROADM Optical Node", "site": "Network Site",
         "wavelength": "Coherent Wavelength", "port": "Client Port", "och": "Optical Channel"}
spec_rows = []
for i, (r, t) in enumerate(zip(src, type_plan)):
    cap = round(float(r.max_bandwidth_mbps) / 1000.0, 2)
    spec_rows.append((int(r.resource_specification_id), f"Lakelink Fiber {int(round(cap))}G {label[t]} v{i%3+1}", t, cap,
                      f"{CAT}.tmf_resource.resource_specification", str(r.resource_specification_id), "MIXED", RUN_ID))
spec_df = spark.createDataFrame(spec_rows, "spec_id BIGINT, name STRING, type STRING, capacity_gbps DOUBLE, source_table STRING, source_id STRING, origin STRING, simulation_run_id STRING")
publish("resource_specification", spec_df,
    "Lakelink Fiber Network Scenario Twin: resource specification catalog. Grain: 1 row per spec (20 rows). EXTEND of tmf_resource.resource_specification; origin MIXED; run nst-sim-001. No injected anomaly.",
    {"spec_id": "PK. Resource specification id; equals tmf_resource.resource_specification.resource_specification_id.",
     "name": "Lakelink Fiber display name of the spec (no vendor branding).",
     "type": "Twin spec type: fiber_span, optical_node, site, wavelength, port, och.",
     "capacity_gbps": "Nominal capacity in Gbps = source max_bandwidth_mbps / 1000.", **LINEAGE})

In [0]:
# Ratified reference file (data/corridor_reference.csv); endpoints FK -> fiber_capital_intelligence.geographic_site
with open(f"{ROOT}/data/corridor_reference.csv") as f:
    rows = list(csv.DictReader(f))
nz = lambda v: v if v not in ("", None) else None
fl = lambda v: float(v) if nz(v) else None
cor_rows = [(r["corridor_id"], r["corridor_name"], r["route_type"], int(r["site_a_id"]), int(r["site_z_id"]), fl(r["distance_km"]),
             fl(r["source_distance_km"]), int(r["fiber_count"]), fl(r["capacity_gbps"]), nz(r["diverse_of"]), fl(r["baseline_util_pct"]),
             fl(r["end_util_pct"]), nz(r["injected_anomaly"]), nz(r["source_table"]), nz(r["source_id"]), r["origin"], RUN_ID) for r in rows]
cor_df = spark.createDataFrame(cor_rows,
    "corridor_id STRING, corridor_name STRING, route_type STRING, site_a_id INT, site_z_id INT, distance_km DOUBLE, source_distance_km DOUBLE, "
    "fiber_count INT, capacity_gbps DOUBLE, diverse_of STRING, baseline_util_pct DOUBLE, end_util_pct DOUBLE, injected_anomaly STRING, "
    "source_table STRING, source_id STRING, origin STRING, simulation_run_id STRING")
publish("corridor", cor_df,
    "Lakelink Fiber Network Scenario Twin: fiber corridors (long-haul and metro). Grain: 1 row per corridor (40 rows: 33 MIXED from fiber_capital_intelligence.silver_corridor, 7 SYNTHETIC). Injected anomalies: DEN-SLC RapidRoute exhaustion (55% to 85% utilization, ~120 days to 95%), CHI-DEN / CHI-DEN-D2 sold as diverse but sharing SRLG-88.",
    {"corridor_id": "PK. Corridor id: <A>-<Z> alphabetical city codes, <CITY>-MET for metro rings, <base>-D2 for diverse routes.",
     "corridor_name": "Lakelink Fiber corridor display name (DEN-SLC = DEN-SLC RapidRoute).",
     "route_type": "long_haul or metro.",
     "site_a_id": "FK -> fiber_capital_intelligence.geographic_site.geographic_site_id (A endpoint).",
     "site_z_id": "FK -> fiber_capital_intelligence.geographic_site.geographic_site_id (Z endpoint).",
     "distance_km": "Route length in km. Long-haul = great-circle x 1.25 (diverse -D2 x1.12 more); metro = source value.",
     "source_distance_km": "Raw silver_corridor.total_distance_km (sum of segment lengths). Lineage only; do not use for analysis.",
     "fiber_count": "Fiber strand count on the corridor.",
     "capacity_gbps": "Corridor lit capacity in Gbps (DEN-SLC = 9600).",
     "diverse_of": "FK -> corridor.corridor_id of the primary route this diverse route protects; null otherwise.",
     "baseline_util_pct": "Generator target utilization (0-100) at 2026-08-18.",
     "end_util_pct": "Generator target utilization (0-100) at 2026-11-16.",
     "injected_anomaly": "Demo anomaly label: DEN-SLC_EXHAUSTION or SRLG-88_DIVERSE_PAIR; null otherwise.", **LINEAGE})

In [0]:
# customer_id drawn ONLY from the audited pool (P1-CUSTOMER-NAME-AUDIT); fictional display names, deterministic seed
pool = spark.table(FQ("dq_customer_candidate_pool")).orderBy("customer_id").collect()
rng = random.Random(SEED)
picked = sorted(rng.sample(pool, 120), key=lambda r: r.customer_id)
ADJ = ["Aspen","Bluestem","Cedar Ridge","Copperleaf","Driftwood","Ember","Foxglove","Granite Peak","Harborline","Ironwood","Juniper","Kestrel",
       "Larkspur","Meridian Plains","Northwind","Oakhollow","Prairie Sky","Quarry Hill","Redfern","Silverbrook","Tamarack","Umber","Valewood",
       "Willowmere","Yarrow","Zephyr","Amberfield","Briar Creek","Cobalt Mesa","Dunmore","Elmstead","Fernhaven","Glacier Run","Highgate"]
NOUN = ["Logistics","Health Partners","Data Systems","Manufacturing","Credit Union","Media Group","Analytics","Energy Cooperative",
        "Freight","Biosciences","Retail Group","Learning Network","Water District","Hospitality","Robotics","Insurance Mutual",
        "Cloud Works","Foods","Engineering","Capital Partners"]
SUF = ["", " Co.", " Group", " Holdings", " Collective"]
DENY = ["lakelink","lumen","centurylink","level 3","att","at&t","verizon","comcast","zayo","cogent","google","amazon","microsoft","meta",
        "apple","ibm","oracle","cisco","t-mobile","sprint","charter","frontier","windstream","crown castle","equinix","nvidia"]
names, cust = set(), []
for r in picked:
    while True:
        n = f"{rng.choice(ADJ)} {rng.choice(NOUN)}{rng.choice(SUF)}"
        if n not in names and not any(d in n.lower() for d in DENY if len(d) > 3) and not any(w in DENY for w in n.lower().split()):
            break
    names.add(n); cust.append((r, n))
ids = [r.customer_id for r, _ in cust]
dual = set(rng.sample(ids, 12))                                   # SRLG-88 Dual Path customers
iowa = set(rng.sample([i for i in ids if i not in dual], 3))       # Iowa wire center IA-WC-01 high-value customers
remaining = [i for i in ids if i not in dual and i not in iowa]
hyper = set(rng.sample(remaining, 6)); gov = set(rng.sample([i for i in remaining if i not in hyper], 8))
def seg(r):
    if r.customer_id in hyper: return "hyperscaler"
    if r.customer_id in gov: return "government"
    if r.type == "wholesale": return "wholesale"
    return "smb" if r.segment_classification == "sme" else "enterprise"
cust_rows = []
for r, n in cust:
    s, cid = seg(r), r.customer_id
    naas = cid in dual or s in ("hyperscaler", "enterprise") and rng.random() < 0.45 or rng.random() < 0.15
    cust_rows.append((cid, n, s, f"{r.segment_classification}/{r.type}", bool(naas), cid in dual,
                      "IA-WC-01" if cid in iowa else None, cid in iowa, r.source_name_code,
                      f"{CAT}.tmf_customer.customer", str(cid), "MIXED", RUN_ID))
cust_df = spark.createDataFrame(cust_rows,
    "customer_id BIGINT, name STRING, segment STRING, source_segment STRING, naas_flag BOOLEAN, dual_path_flag BOOLEAN, copper_wc_id STRING, "
    "high_value_flag BOOLEAN, source_name_code STRING, source_table STRING, source_id STRING, origin STRING, simulation_run_id STRING")
publish("customer", cust_df,
    "Lakelink Fiber Network Scenario Twin: business customers. Grain: 1 row per customer (120 rows). customer_id drawn only from dq_customer_candidate_pool (-> tmf_customer.customer); names are fictional. Injected: 12 Dual Path customers (dual_path_flag) on SRLG-88 spans; 3 high-value customers on Iowa wire center IA-WC-01.",
    {"customer_id": "PK and FK -> tmf_customer.customer.customer_id (via dq_customer_candidate_pool).",
     "name": "Fictional business display name (deterministic, seed 4471, checked against a real-brand denylist). Never a real company.",
     "segment": "Twin segment: enterprise, wholesale, hyperscaler, government, smb.",
     "source_segment": "Source segment_classification/type from tmf_customer.customer, kept for lineage.",
     "naas_flag": "TRUE if the customer buys NaaS / on-demand services.",
     "dual_path_flag": "TRUE for the 12 Dual Path customers whose diverse paths ride S-4471 and S-4472 (shared SRLG-88).",
     "copper_wc_id": "FK -> copper_wire_center.wc_id; set only for the 3 Iowa high-value customers (IA-WC-01).",
     "high_value_flag": "High-value customer. TRUE for the 3 Iowa customers now; top-decile MRR customers added in P1-GEN-SERVICE.",
     "source_name_code": "Opaque source name code from tmf_customer.customer (lineage only; not for display).", **LINEAGE})

In [0]:
display(spark.sql(f"""
SELECT 'spec_rows' k, count(*) v FROM {FQ('resource_specification')}
UNION ALL SELECT 'spec_fk_orphan', count(*) FROM {FQ('resource_specification')} s LEFT ANTI JOIN {CAT}.tmf_resource.resource_specification p ON s.spec_id=p.resource_specification_id
UNION ALL SELECT 'cor_rows', count(*) FROM {FQ('corridor')}
UNION ALL SELECT 'cor_site_orphan', count(*) FROM (SELECT site_a_id s FROM {FQ('corridor')} UNION ALL SELECT site_z_id FROM {FQ('corridor')}) c LEFT ANTI JOIN {CAT}.fiber_capital_intelligence.geographic_site g ON c.s=g.geographic_site_id
UNION ALL SELECT 'denslc_rapidroute', count(*) FROM {FQ('corridor')} WHERE corridor_id='DEN-SLC' AND corridor_name='DEN-SLC RapidRoute' AND capacity_gbps=9600
UNION ALL SELECT 'cust_rows', count(*) FROM {FQ('customer')}
UNION ALL SELECT 'cust_parent_orphan', count(*) FROM {FQ('customer')} c LEFT ANTI JOIN {CAT}.tmf_customer.customer p USING (customer_id)
UNION ALL SELECT 'dual_path', count_if(dual_path_flag) FROM {FQ('customer')}
UNION ALL SELECT 'iowa_hv', count_if(copper_wc_id='IA-WC-01' AND high_value_flag) FROM {FQ('customer')}
"""))

## Next sections
* **P1-GEN-TOPOLOGY**: physical_resource (800; S-4471 on CHI-DEN, S-4472 on CHI-DEN-D2, both SRLG-88), logical_resource (2,500)
* **P1-GEN-SERVICE**: service + service_level_agreement (1,500 each); route each dual_path_flag customer primary over S-4471 and protect over S-4472; then set high_value_flag for top-decile MRR customers
* **P1-GEN-TELEMETRY**: telemetry_span_hourly (1,296,000; DEN-SLC curve)
* **P1-GEN-OPS**: alarm, trouble_ticket, maintenance_window (S-4471 window 2026-11-20 06:00 UTC)
* **P1-GEN-COPPER**: copper_wire_center (60; IA-WC-01 = the flagged Iowa WC, high_value_customers=3, target 2027-05-01)